# 逻辑回归

在本练习中，你将实现逻辑回归并将其应用于两个不同的数据集。


# 大纲
- [ 1 - 软件包 ](#1)
- [ 2 - 逻辑回归](#2)
  - [ 2.1 问题陈述](#2.1)
  - [ 2.2 加载和可视化数据](#2.2)
  - [ 2.3  Sigmoid 函数](#2.3)
  - [ 2.4 逻辑回归的代价函数](#2.4)
  - [ 2.5 逻辑回归的梯度](#2.5)
  - [ 2.6 使用梯度下降学习参数 ](#2.6)
  - [ 2.7 绘制决策边界](#2.7)
  - [ 2.8 评估逻辑回归](#2.8)
- [ 3 - 正则化逻辑回归](#3)
  - [ 3.1 问题陈述](#3.1)
  - [ 3.2 加载和可视化数据](#3.2)
  - [ 3.3 特征映射](#3.3)
  - [ 3.4 正则化逻辑回归的代价函数](#3.4)
  - [ 3.5 正则化逻辑回归的梯度](#3.5)
  - [ 3.6 使用梯度下降学习参数](#3.6)
  - [ 3.7 绘制决策边界](#3.7)
  - [ 3.8 评估正则化逻辑回归模型](#3.8)


_**注意：** 为防止自动评分器出错，本实验中不允许编辑或删除非评分单元格。请也不要添加任何新单元格。
**通过本作业后**，如果你想尝试任何非评分代码，可以按照本笔记本底部的说明操作。_

<a name="1"></a>
## 1 - 软件包 

首先，运行下面的单元格以导入本次作业所需的所有软件包。
- [numpy](www.numpy.org) 是使用 Python 进行科学计算的基础软件包。
- [matplotlib](http://matplotlib.org) 是一个用于在 Python 中绘制图表的著名库。
-  ``utils.py`` 包含本作业的辅助函数。你不需要修改此文件中的代码。

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
import copy
import math

%matplotlib inline

<a name="2"></a>
## 2 - 逻辑回归

在本练习的这一部分，你将构建一个逻辑回归模型来预测学生是否被大学录取。

<a name="2.1"></a>
### 2.1 问题陈述

假设你是一所大学院系的管理员，你想根据申请人在两门考试中的成绩来确定每位申请人的录取机会。
* 你有以往申请人的历史数据，可以将其用作逻辑回归的训练集。
* 对于每个训练样本，你有申请人在两门考试中的分数以及录取决定。
* 你的任务是构建一个分类模型，根据这两门考试的分数估算申请人被录取的概率。

<a name="2.2"></a>
### 2.2 加载和可视化数据

你将首先加载此任务的数据集。
- 下面显示的 `load_dataset()` 函数将数据加载到变量 `X_train` 和 `y_train` 中
  - `X_train` 包含学生在两门考试中的分数
  - `y_train` 是录取决定
      - 如果学生被录取，则 `y_train = 1`
      - 如果学生未被录取，则 `y_train = 0`
  - `X_train` 和 `y_train` 都是 numpy 数组。


In [ ]:
# 加载数据集
X_train, y_train = load_data("data/ex2data1.txt")

#### 查看变量
让我们更熟悉一下你的数据集。
- 一个好的起点是打印出每个变量，看看它包含什么。

下面的代码打印 `X_train` 的前五个值和变量的类型。

In [ ]:
print("First five elements in X_train are:\n", X_train[:5])
print("Type of X_train:",type(X_train))

现在打印 `y_train` 的前五个值

In [ ]:
print("First five elements in y_train are:\n", y_train[:5])
print("Type of y_train:",type(y_train))

#### 检查变量的维度

另一种熟悉数据的有用方法是查看其维度。让我们打印 `X_train` 和 `y_train` 的形状，看看数据集中有多少训练样本。

In [ ]:
print ('The shape of X_train is: ' + str(X_train.shape))
print ('The shape of y_train is: ' + str(y_train.shape))
print ('We have m = %d training examples' % (len(y_train)))

#### 可视化数据

在开始实现任何学习算法之前，如果可能的话，可视化数据总是好的。
- 下面的代码在 2D 图上显示数据（如下所示），其中坐标轴是两门考试的分数，正例和负例用不同的标记显示。
- 我们使用 ``utils.py`` 文件中的辅助函数来生成此图。

<img src="images/figure 1.png" width="450" height="450">



In [ ]:
# 绘制样本
plot_data(X_train, y_train[:], pos_label="Admitted", neg_label="Not admitted")

# 设置 y 轴标签
plt.ylabel('Exam 2 score') 
# 设置 x 轴标签
plt.xlabel('Exam 1 score') 
plt.legend(loc="upper right")
plt.show()

你的目标是构建一个逻辑回归模型来拟合这些数据。
- 有了这个模型，你就可以根据学生在两门考试中的分数预测新学生是否会被录取。

<a name="2.3"></a>
### 2.3  Sigmoid 函数

回想一下，对于逻辑回归，模型表示为

$$ f_{\mathbf{w},b}(x) = g(\mathbf{w}\cdot \mathbf{x} + b)$$
其中函数 $g$ 是 sigmoid 函数。Sigmoid 函数定义为：

$$g(z) = \frac{1}{1+e^{-z}}$$

让我们先实现 sigmoid 函数，以便本作业的其余部分可以使用它。

<a name='ex-01'></a>
### 练习 1
请完成 `sigmoid` 函数以计算

$$g(z) = \frac{1}{1+e^{-z}}$$

注意
- `z` 并不总是单个数字，也可以是数字数组。
- 如果输入是数字数组，我们希望将 sigmoid 函数应用于输入数组中的每个值。

如果你遇到困难，可以查看下面单元格后面的提示来帮助你实现。

In [ ]:
# UNQ_C1
# 评分函数：sigmoid

def sigmoid(z):
    """
    计算 z 的 sigmoid 值

    参数:
        z (ndarray): 标量或任意大小的 numpy 数组。

    返回:
        g (ndarray): sigmoid(z)，与 z 形状相同
         
    """
          
    ### 开始编码 ### 
    
    ### 结束编码 ###  
    
    return g

<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
       
`numpy` 有一个名为 [`np.exp()`](https://numpy.org/doc/stable/reference/generated/numpy.exp.html) 的函数，它提供了一种便捷的方式来计算输入数组 (`z`) 中所有元素的指数 ($e^{z}$)。
 
<details>
          <summary><font size="2" color="darkblue"><b> 点击获取更多提示</b></font></summary>
        
  - 你可以将 $e^{-z}$ 翻译为代码 `np.exp(-z)`
    
  - 你可以将 $1/e^{-z}$ 翻译为代码 `1/np.exp(-z)`
    
    如果你仍然卡住，可以查看下面的提示来计算 `g`
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 g 的提示</b></font></summary>
        <code>g = 1 / (1 + np.exp(-z))</code>
    </details>


</details>

完成后，尝试在下面的单元格中调用 `sigmoid(x)` 来测试几个值。
- 对于大的正 x 值，sigmoid 应该接近 1，而对于大的负值，sigmoid 应该接近 0。
- 评估 `sigmoid(0)` 应该正好得到 0.5。


In [ ]:
# 注意：你可以编辑这个值
value = 0

print (f"sigmoid({value}) = {sigmoid(value)}")

**期望输出**：
<table>
  <tr>
    <td> <b>sigmoid(0)<b></td>
    <td> 0.5 </td> 
  </tr>
</table>
    
- 如前所述，你的代码也应该适用于向量和矩阵。对于矩阵，你的函数应该对每个元素执行 sigmoid 函数。

In [ ]:
print ("sigmoid([ -1, 0, 1, 2]) = " + str(sigmoid(np.array([-1, 0, 1, 2]))))

# 单元测试
from public_tests import *
sigmoid_test(sigmoid)

**期望输出**：
<table>
  <tr>
    <td><b>sigmoid([-1, 0, 1, 2])<b></td> 
    <td>[0.26894142        0.5           0.73105858        0.88079708]</td> 
  </tr>    
  
</table>

<a name="2.4"></a>
### 2.4 逻辑回归的代价函数

在本节中，你将实现逻辑回归的代价函数。

<a name='ex-02'></a>
### 练习 2

请使用下面的方程完成 `compute_cost` 函数。

回想一下，对于逻辑回归，代价函数的形式为

$$ J(\mathbf{w},b) = \frac{1}{m}\sum_{i=0}^{m-1} \left[ loss(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)}) \right] \tag{1}$$

其中
* m 是数据集中训练样本的数量


* $loss(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)})$ 是单个数据点的代价，即 -

    $$loss(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)}) = (-y^{(i)} \log\left(f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) - \left( 1 - y^{(i)}\right) \log \left( 1 - f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) \tag{2}$$
    
    
*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是模型的预测值，而 $y^{(i)}$ 是实际标签

*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = g(\mathbf{w} \cdot \mathbf{x^{(i)}} + b)$ 其中函数 $g$ 是 sigmoid 函数。
    * 首先计算中间变量 $z_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x^{(i)}} + b = w_0x^{(i)}_0 + ... + w_{n-1}x^{(i)}_{n-1} + b$（其中 $n$ 是特征数量）可能会有所帮助，然后再计算 $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = g(z_{\mathbf{w},b}(\mathbf{x}^{(i)}))$

注意：
* 在执行此操作时，请记住变量 `X_train` 和 `y_train` 不是标量值，而是形状分别为 ($m, n$) 和 ($𝑚$,1) 的矩阵，其中 $𝑛$ 是特征数量，$𝑚$ 是训练样本数量。
* 你可以使用上面实现的 sigmoid 函数来完成此部分。

如果你遇到困难，可以查看下面单元格后面的提示来帮助你实现。

In [ ]:
# UNQ_C2
# 评分函数：compute_cost
def compute_cost(X, y, w, b, lambda_= 1):
    """
    计算所有样本的代价
    参数:
      X : (ndarray 形状 (m,n)) 数据，m 个样本，n 个特征
      y : (array_like 形状 (m,)) 目标值
      w : (array_like 形状 (n,)) 模型参数值
      b : 标量，模型的偏置参数值
      lambda_: 未使用的占位符
    返回:
      total_cost: (标量) 代价
    """

    m, n = X.shape
    
    ### 开始编码 ###
    
    
        
        
            
        
        
        
        
    
    ### 结束编码 ### 

    return total_cost

<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
    
    
   * 你可以用代码表示求和运算符，例如：$h = \sum\limits_{i = 0}^{m-1} 2i$ 可以表示为：
    ```python 
        h = 0
        for i in range(m):
            h = h + 2*i
    ```
  
   * 在这种情况下，你可以使用 for 循环遍历 `X` 中的所有样本，并将每次迭代的 `loss` 添加到循环外初始化的变量 (`loss_sum`) 中。

   * 然后，你可以将 `total_cost` 返回为 `loss_sum` 除以 `m`。
     
    <details>
          <summary><font size="2" color="darkblue"><b> 点击获取更多提示</b></font></summary>
        
    * 以下是此函数的整体实现结构
    ```python 
    def compute_cost(X, y, w, b, lambda_= 1):
        m, n = X.shape
    
        ### 开始编码 ###
        loss_sum = 0 
        
        # 遍历每个训练样本
        for i in range(m): 
            
            # 首先计算 z_wb = w[0]*X[i][0]+...+w[n-1]*X[i][n-1]+b
            z_wb = 0 
            # 遍历每个特征
            for j in range(n): 
                # 将相应的项添加到 z_wb
                z_wb_ij = # 你的代码，计算 w[j] * X[i][j]
                z_wb += z_wb_ij # 等价于 z_wb = z_wb + z_wb_ij
            # 将偏置项添加到 z_wb
            z_wb += b # 等价于 z_wb = z_wb + b
        
            f_wb = # 你的代码，计算训练样本的预测值 f_wb
            loss =  # 你的代码，计算训练样本的损失
            
            loss_sum += loss # 等价于 loss_sum = loss_sum + loss
        
        total_cost = (1 / m) * loss_sum  
        ### 结束编码 ### 
        
        return total_cost
    ```
    
    如果你仍然卡住，可以查看下面的提示来计算 `z_wb_ij`、`f_wb` 和 `cost`。
    <details>
          <summary><font size="2" color="darkblue"><b>计算 z_wb_ij 的提示</b></font></summary>
           &emsp; &emsp; <code>z_wb_ij = w[j]*X[i][j] </code>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>计算 f_wb 的提示</b></font></summary>
           &emsp; &emsp; $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = g(z_{\mathbf{w},b}(\mathbf{x}^{(i)}))$ 其中 $g$ 是 sigmoid 函数。你可以简单地调用上面实现的 `sigmoid` 函数。
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 f 的更多提示</b></font></summary>
               &emsp; &emsp; 你可以将 f_wb 计算为 <code>f_wb = sigmoid(z_wb) </code>
           </details>
    </details>

     <details>
          <summary><font size="2" color="darkblue"><b>计算 loss 的提示</b></font></summary>
          &emsp; &emsp; 你可以使用 <a href="https://numpy.org/doc/stable/reference/generated/numpy.log.html">np.log</a> 函数来计算对数
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 loss 的更多提示</b></font></summary>
              &emsp; &emsp; 你可以将 loss 计算为 <code>loss =  -y[i] * np.log(f_wb) - (1 - y[i]) * np.log(1 - f_wb)</code>
          </details>
    </details>
        
    </details>

</details>

运行下面的单元格，使用参数 $w$ 和 $b$ 的两种不同初始化来检查你实现的 `compute_cost` 函数

In [ ]:
m, n = X_train.shape

# 计算并显示 w 和 b 初始化为零时的代价
initial_w = np.zeros(n)
initial_b = 0.
cost = compute_cost(X_train, y_train, initial_w, initial_b)
print('Cost at initial w and b (zeros): {:.3f}'.format(cost))

**期望输出**：
<table>
  <tr>
    <td> <b>Cost at initial w and b (zeros)<b></td>
    <td> 0.693 </td> 
  </tr>
</table>

In [ ]:
# 计算并显示非零 w 和 b 时的代价
test_w = np.array([0.2, 0.2])
test_b = -24.
cost = compute_cost(X_train, y_train, test_w, test_b)

print('Cost at test w and b (non-zeros): {:.3f}'.format(cost))


# 单元测试
compute_cost_test(compute_cost)

**期望输出**：
<table>
  <tr>
    <td> <b>Cost at test w and b (non-zeros):<b></td>
    <td> 0.218 </td> 
  </tr>
</table>

<a name="2.5"></a>
### 2.5 逻辑回归的梯度

在本节中，你将实现逻辑回归的梯度。

回想一下梯度下降算法是：

$$\begin{align*}& \text{repeat until convergence:} \; \lbrace \newline \; & b := b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b} \newline       \; & w_j := w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{1}  \; & \text{for j := 0..n-1}\newline & \rbrace\end{align*}$$

其中，参数 $b$、$w_j$ 都是同时更新的


<a name='ex-03'></a>
### 练习 3

请完成 `compute_gradient` 函数，根据下面的方程 (2) 和 (3) 计算 $\frac{\partial J(\mathbf{w},b)}{\partial w}$、$\frac{\partial J(\mathbf{w},b)}{\partial b}$。

$$
\frac{\partial J(\mathbf{w},b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - \mathbf{y}^{(i)}) \tag{2}
$$
$$
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - \mathbf{y}^{(i)})x_{j}^{(i)} \tag{3}
$$
* m 是数据集中训练样本的数量

    
*  $f_{\mathbf{w},b}(x^{(i)})$ 是模型的预测值，而 $y^{(i)}$ 是实际标签


- **注意**：虽然这个梯度看起来与线性回归梯度相同，但公式实际上是不同的，因为线性回归和逻辑回归对 $f_{\mathbf{w},b}(x)$ 的定义不同。

和之前一样，你可以使用上面实现的 sigmoid 函数，如果你遇到困难，可以查看下面单元格后面的提示来帮助你实现。

In [ ]:
# UNQ_C3
# 评分函数：compute_gradient
def compute_gradient(X, y, w, b, lambda_=None): 
    """
    计算逻辑回归的梯度
 
    参数:
      X : (ndarray 形状 (m,n)) 变量，如房屋大小
      y : (array_like 形状 (m,1)) 实际值
      w : (array_like 形状 (n,1)) 模型参数值
      b : (标量) 模型参数值
      lambda_: 未使用的占位符。
    返回:
      dj_dw: (array_like 形状 (n,1)) 代价关于参数 w 的梯度。
      dj_db: (标量) 代价关于参数 b 的梯度。
    """
    m, n = X.shape
    dj_dw = np.zeros(w.shape)
    dj_db = 0.

    ### 开始编码 ### 
    for i in range(m):
        z_wb = None
        for j in range(n): 
            z_wb += None
        z_wb += None
        f_wb = None
        
        dj_db_i = None
        dj_db += None
        
        for j in range(n):
            dj_dw[j] = None
            
    dj_dw = None
    dj_db = None
    ### 结束编码 ###

        
    return dj_db, dj_dw

 <details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
    
    
* 以下是此函数的整体实现结构
    ```python 
       def compute_gradient(X, y, w, b, lambda_=None): 
            m, n = X.shape
            dj_dw = np.zeros(w.shape)
            dj_db = 0.
        
            ### 开始编码 ### 
            for i in range(m):
                # 计算 f_wb（与上面 compute_cost 函数中的方法完全相同）
                f_wb = 
        
                # 从这个样本计算 b 的梯度
                dj_db_i = # 你的代码，计算误差
        
                # 将其添加到 dj_db
                dj_db += dj_db_i
        
                # 获取每个属性的 dj_dw
                for j in range(n):
                    # 你的代码，计算第 i 个样本对第 j 个属性的梯度
                    dj_dw_ij =  
                    dj_dw[j] += dj_dw_ij
        
            # 将 dj_db 和 dj_dw 除以总样本数
            dj_dw = dj_dw / m
            dj_db = dj_db / m
            ### 结束编码 ###
       
            return dj_db, dj_dw
    ```
  
    如果你仍然卡住，可以查看下面的提示来计算 `f_wb`、`dj_db_i` 和 `dj_dw_ij`
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 f_wb 的提示</b></font></summary>
           &emsp; &emsp; 回想一下你在上面的 <code>compute_cost</code> 中计算了 f_wb — 有关如何计算每个中间项的详细提示，请查看该练习下面的提示部分
           <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 f_wb 的更多提示</b></font></summary>
              &emsp; &emsp; 你可以将 f_wb 计算为
               <pre>
               for i in range(m):   
                   # 计算 f_wb（与上面 compute_cost 函数中的方法完全相同）
                   z_wb = 0
                   # 遍历每个特征
                   for j in range(n): 
                       # 将相应的项添加到 z_wb
                       z_wb_ij = X[i, j] * w[j]
                       z_wb += z_wb_ij
            
                   # 添加偏置项
                   z_wb += b
        
                   # 计算模型的预测值
                   f_wb = sigmoid(z_wb)
    </details>
        
    </details>
    <details>
          <summary><font size="2" color="darkblue"><b>计算 dj_db_i 的提示</b></font></summary>
           &emsp; &emsp; 你可以将 dj_db_i 计算为 <code>dj_db_i = f_wb - y[i]</code>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>计算 dj_dw_ij 的提示</b></font></summary>
        &emsp; &emsp; 你可以将 dj_dw_ij 计算为 <code>dj_dw_ij = (f_wb - y[i])* X[i][j]</code>
    </details>

</details>

运行下面的单元格，使用参数 $w$ 和 $b$ 的两种不同初始化来检查你实现的 `compute_gradient` 函数

In [ ]:
# 计算并显示 w 和 b 初始化为零时的梯度
initial_w = np.zeros(n)
initial_b = 0.

dj_db, dj_dw = compute_gradient(X_train, y_train, initial_w, initial_b)
print(f'dj_db at initial w and b (zeros):{dj_db}' )
print(f'dj_dw at initial w and b (zeros):{dj_dw.tolist()}' )

**期望输出**：
<table>
  <tr>
    <td> <b>dj_db at initial w and b (zeros)<b></td>
    <td> -0.1 </td> 
  </tr>
  <tr>
    <td> <b>dj_dw at initial w and b (zeros):<b></td>
    <td> [-12.00921658929115, -11.262842205513591] </td> 
  </tr>
</table>

In [ ]:
# 计算并显示非零 w 和 b 时的代价和梯度
test_w = np.array([ 0.2, -0.5])
test_b = -24
dj_db, dj_dw  = compute_gradient(X_train, y_train, test_w, test_b)

print('dj_db at test w and b:', dj_db)
print('dj_dw at test w and b:', dj_dw.tolist())

# 单元测试
compute_gradient_test(compute_gradient)

**期望输出**：
<table>
  <tr>
    <td> <b>dj_db at test w and b (non-zeros)<b></td>
    <td> -0.5999999999991071 </td> 
  </tr>
  <tr>
    <td> <b>dj_dw at test w and b (non-zeros):<b></td>
    <td>  [-44.8313536178737957, -44.37384124953978] </td> 
  </tr>
</table>

<a name="2.6"></a>
### 2.6 使用梯度下降学习参数 

与之前的作业类似，你现在将使用梯度下降来找到逻辑回归模型的最优参数。
- 这部分你不需要实现任何东西。只需运行下面的单元格。

- 验证梯度下降是否正常工作的一个好方法是查看 $J(\mathbf{w},b)$ 的值，并检查它是否在每一步都在减小。

- 假设你正确实现了梯度并计算了代价，你的 $J(\mathbf{w},b)$ 值应该永远不会增加，并且应该在算法结束时收敛到一个稳定值。

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters, lambda_): 
    """
    执行批量梯度下降来学习 theta。通过执行 num_iters 次梯度下降步骤（学习率为 alpha）来更新 theta
    
    参数:
      X :    (array_like 形状 (m, n)
      y :    (array_like 形状 (m,))
      w_in : (array_like 形状 (n,))  模型参数的初始值
      b_in : (标量)                 模型参数的初始值
      cost_function:                 计算代价的函数
      alpha : (浮点数)               学习率
      num_iters : (整数)             运行梯度下降的迭代次数
      lambda_ (标量, 浮点数)         正则化常数
      
    返回:
      w : (array_like 形状 (n,)) 运行梯度下降后模型参数的更新值
      b : (标量)                运行梯度下降后模型参数的更新值
    """
    
    # 训练样本数量
    m = len(X)
    
    # 用于存储每次迭代的代价 J 和 w 的数组，主要用于后续绘图
    J_history = []
    w_history = []
    
    for i in range(num_iters):

        # 计算梯度并更新参数
        dj_db, dj_dw = gradient_function(X, y, w_in, b_in, lambda_)   

        # 使用 w、b、alpha 和梯度更新参数
        w_in = w_in - alpha * dj_dw               
        b_in = b_in - alpha * dj_db              
       
        # 在每次迭代时保存代价 J
        if i<100000:      # 防止资源耗尽
            cost =  cost_function(X, y, w_in, b_in, lambda_)
            J_history.append(cost)

        # 每隔 10 次迭代或迭代次数少于 10 时打印代价
        if i% math.ceil(num_iters/10) == 0 or i == (num_iters-1):
            w_history.append(w_in)
            print(f"Iteration {i:4}: Cost {float(J_history[-1]):8.2f}   ")
        
    return w_in, b_in, J_history, w_history #返回 w 和 J,w 历史用于绘图

现在让我们运行上面的梯度下降算法来学习数据集的参数。

**注意**

下面的代码块需要几分钟才能运行，特别是非向量化版本。你可以减少 `iterations` 来测试你的实现并更快地迭代。如果你有时间，可以尝试运行 100,000 次迭代以获得更好的结果。

In [ ]:
np.random.seed(1)
intial_w = 0.01 * (np.random.rand(2).reshape(-1,1) - 0.5)
initial_b = -8


# 梯度下降设置
iterations = 10000
alpha = 0.001

w,b, J_history,_ = gradient_descent(X_train ,y_train, initial_w, initial_b, 
                                   compute_cost, compute_gradient, alpha, iterations, 0)

<details>
<summary>
    <b>期望输出：代价 0.30，（点击查看详情）：</b>
</summary>

    # 使用以下设置
    np.random.seed(1)
    intial_w = 0.01 * (np.random.rand(2).reshape(-1,1) - 0.5)
    initial_b = -8
    iterations = 10000
    alpha = 0.001
    #

```
Iteration    0: Cost     1.01   
Iteration 1000: Cost     0.31   
Iteration 2000: Cost     0.30   
Iteration 3000: Cost     0.30   
Iteration 4000: Cost     0.30   
Iteration 5000: Cost     0.30   
Iteration 6000: Cost     0.30   
Iteration 7000: Cost     0.30   
Iteration 8000: Cost     0.30   
Iteration 9000: Cost     0.30   
Iteration 9999: Cost     0.30   
```

<a name="2.7"></a>
### 2.7 绘制决策边界

我们现在将使用梯度下降的最终参数来绘制线性拟合。如果你正确实现了前面的部分，你应该看到下面的图：
<img src="images/figure 2.png"  width="450" height="450">

我们将使用 `utils.py` 文件中的辅助函数来创建此图。

In [ ]:
plot_decision_boundary(w, b, X_train, y_train)

<a name="2.8"></a>
### 2.8 评估逻辑回归

我们可以通过观察学习到的模型在训练集上的预测效果来评估我们找到的参数的质量。

你将在下面实现 `predict` 函数来完成此操作。


<a name='ex-04'></a>
### 练习 4

请完成 `predict` 函数，给定数据集和学习到的参数向量 $w$ 和 $b$，产生 `1` 或 `0` 的预测。
- 首先你需要对每个样本计算模型的预测 $f(x^{(i)}) = g(w \cdot x^{(i)} + b)$
    - 你在上面的部分已经实现了这个
- 我们将模型的输出 ($f(x^{(i)})$) 解释为给定 $x^{(i)}$ 并以 $w$ 为参数时 $y^{(i)}=1$ 的概率。
- 因此，要从逻辑回归模型获得最终预测 ($y^{(i)}=0$ 或 $y^{(i)}=1$)，你可以使用以下启发式方法 -

  如果 $f(x^{(i)}) >= 0.5$，预测 $y^{(i)}=1$
  
  如果 $f(x^{(i)}) < 0.5$，预测 $y^{(i)}=0$
    
如果你遇到困难，可以查看下面单元格后面的提示来帮助你实现。

In [ ]:
# UNQ_C4
# 评分函数：predict

def predict(X, w, b): 
    """
    使用学习到的逻辑回归参数 w 预测标签是 0 还是 1
    
    参数:
    X : (ndarray 形状 (m, n))
    w : (array_like 形状 (n,))      模型参数
    b : (标量, 浮点数)              模型参数

    返回:
    p: (ndarray (m,1))
        使用 0.5 作为阈值对 X 的预测
    """
    # 训练样本数量
    m, n = X.shape   
    p = np.zeros(m)
   
    ### 开始编码 ### 
    # 遍历每个样本
    for i in range(m):   
        z_wb = None
        # 遍历每个特征
        for j in range(n): 
            # 将相应的项添加到 z_wb
            z_wb += None
        
        # 添加偏置项
        z_wb += None
        
        # 计算此样本的预测值
        f_wb = None

        # 应用阈值
        p[i] = None
        
    ### 结束编码 ### 
    return p

<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
    
    
* 以下是此函数的整体实现结构
    ```python 
       def predict(X, w, b): 
            # 训练样本数量
            m, n = X.shape   
            p = np.zeros(m)
   
            ### 开始编码 ### 
            # 遍历每个样本
            for i in range(m):   
                
                # 计算 f_wb（与上面 compute_cost 函数中的方法完全相同）
                # 使用几行代码
                f_wb = 

                # 计算该训练样本的预测值
                p[i] = # 你的代码，根据 f_wb 计算预测值
        
            ### 结束编码 ### 
            return p
    ```
  
    如果你仍然卡住，可以查看下面的提示来计算 `f_wb` 和 `p[i]`
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 f_wb 的提示</b></font></summary>
           &emsp; &emsp; 回想一下你在上面的 <code>compute_cost</code> 中计算了 f_wb — 有关如何计算每个中间项的详细提示，请查看该练习下面的提示部分
           <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 f_wb 的更多提示</b></font></summary>
              &emsp; &emsp; 你可以将 f_wb 计算为
               <pre>
               for i in range(m):   
                   # 计算 f_wb（与上面 compute_cost 函数中的方法完全相同）
                   z_wb = 0
                   # 遍历每个特征
                   for j in range(n): 
                       # 将相应的项添加到 z_wb
                       z_wb_ij = X[i, j] * w[j]
                       z_wb += z_wb_ij
            
                   # 添加偏置项
                   z_wb += b
        
                   # 计算模型的预测值
                   f_wb = sigmoid(z_wb)
    </details>
        
    </details>
    <details>
          <summary><font size="2" color="darkblue"><b>计算 p[i] 的提示</b></font></summary>
           &emsp; &emsp; 例如，如果你想表达如果 y 小于 3 则 x = 1，否则为 0，你可以用代码表示为 <code>x = y < 3</code>。现在对 p[i] 做同样的事情：如果 f_wb >= 0.5 则 p[i] = 1，否则为 0。
           <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 p[i] 的更多提示</b></font></summary>
              &emsp; &emsp; 你可以将 p[i] 计算为 <code>p[i] = f_wb >= 0.5</code>
          </details>
    </details>

</details>

完成 `predict` 函数后，让我们运行下面的代码，通过计算分类器正确预测的百分比来报告分类器的训练准确率。

In [ ]:
# 测试你的 predict 代码
np.random.seed(1)
tmp_w = np.random.randn(2)
tmp_b = 0.3    
tmp_X = np.random.randn(4, 2) - 0.5

tmp_p = predict(tmp_X, tmp_w, tmp_b)
print(f'Output of predict: shape {tmp_p.shape}, value {tmp_p}')

# 单元测试
predict_test(predict)

**期望输出** 

<table>
  <tr>
    <td> <b>Output of predict: shape (4,),value [0. 1. 1. 1.]<b></td>
  </tr>
</table>

现在让我们用它来计算训练集上的准确率

In [ ]:
# 计算训练集上的准确率
p = predict(X_train, w,b)
print('Train Accuracy: %f'%(np.mean(p == y_train) * 100))

<table>
  <tr>
    <td> <b>Train Accuracy (approx):<b></td>
    <td> 92.00 </td> 
  </tr>
</table>

<a name="3"></a>
## 3 - 正则化逻辑回归

在本练习的这一部分，你将实现正则化逻辑回归来预测来自制造工厂的微芯片是否通过质量保证 (QA)。在 QA 期间，每个微芯片都要经过各种测试以确保其功能正常。

<a name="3.1"></a>
### 3.1 问题陈述

假设你是工厂的产品经理，你有某些微芯片在两项不同测试中的测试结果。
- 根据这两项测试，你想确定微芯片应该被接受还是被拒绝。
- 为了帮助你做出决定，你有一个过去微芯片的测试结果数据集，你可以从中构建一个逻辑回归模型。

<a name="3.2"></a>
### 3.2 加载和可视化数据

与本练习的前几部分类似，让我们先加载此任务的数据集并对其进行可视化。

- 下面显示的 `load_dataset()` 函数将数据加载到变量 `X_train` 和 `y_train` 中
  - `X_train` 包含微芯片在两项测试中的测试结果
  - `y_train` 包含 QA 的结果
      - 如果微芯片被接受，则 `y_train = 1`
      - 如果微芯片被拒绝，则 `y_train = 0`
  - `X_train` 和 `y_train` 都是 numpy 数组。

In [ ]:
# 加载数据集
X_train, y_train = load_data("data/ex2data2.txt")

#### 查看变量

下面的代码打印 `X_train` 和 `y_train` 的前五个值以及变量的类型。


In [ ]:
# 打印 X_train
print("X_train:", X_train[:5])
print("Type of X_train:",type(X_train))

# 打印 y_train
print("y_train:", y_train[:5])
print("Type of y_train:",type(y_train))

#### 检查变量的维度

另一种熟悉数据的有用方法是查看其维度。让我们打印 `X_train` 和 `y_train` 的形状，看看数据集中有多少训练样本。

In [ ]:
print ('The shape of X_train is: ' + str(X_train.shape))
print ('The shape of y_train is: ' + str(y_train.shape))
print ('We have m = %d training examples' % (len(y_train)))

#### 可视化数据

辅助函数 `plot_data`（来自 `utils.py`）用于生成类似图 3 的图，其中坐标轴是两项测试的分数，正例 (y = 1, 接受) 和负例 (y = 0, 拒绝) 用不同的标记显示。

<img src="images/figure 3.png"  width="450" height="450">

In [ ]:
# 绘制样本
plot_data(X_train, y_train[:], pos_label="Accepted", neg_label="Rejected")

# 设置 y 轴标签
plt.ylabel('Microchip Test 2') 
# 设置 x 轴标签
plt.xlabel('Microchip Test 1') 
plt.legend(loc="upper right")
plt.show()

图 3 显示，我们的数据集无法通过图中的一条直线将正例和负例分开。因此，直接应用逻辑回归在这个数据集上表现不佳，因为逻辑回归只能找到线性决策边界。


<a name="3.3"></a>
### 3.3 特征映射

更好地拟合数据的一种方法是从每个数据点创建更多特征。在提供的函数 `map_feature` 中，我们将特征映射到 $x_1$ 和 $x_2$ 的所有多项式项，直到六次幂。

$$\mathrm{map\_feature}(x) = 
\left[\begin{array}{c}
x_1\\
x_2\\
x_1^2\\
x_1 x_2\\
x_2^2\\
x_1^3\\
\vdots\\
x_1 x_2^5\\
x_2^6\end{array}\right]$$

通过这种映射，我们的两个特征向量（两项 QA 测试的分数）被转换为 27 维向量。

- 在这个更高维度特征向量上训练的逻辑回归分类器将具有更复杂的决策边界，并且在我们的二维图中绘制时将是非线性的。
- 我们在 utils.py 中为你提供了 `map_feature` 函数。

In [ ]:
print("Original shape of data:", X_train.shape)

mapped_X =  map_feature(X_train[:, 0], X_train[:, 1])
print("Shape after feature mapping:", mapped_X.shape)

让我们也打印 `X_train` 和 `mapped_X` 的第一个元素来看看转换效果。

In [ ]:
print("X_train[0]:", X_train[0])
print("mapped X_train[0]:", mapped_X[0])

虽然特征映射允许我们构建更有表现力的分类器，但它也更容易过拟合。在本练习的接下来的部分，你将实现正则化逻辑回归来拟合数据，并亲自看看正则化如何帮助解决过拟合问题。

<a name="3.4"></a>
### 3.4 正则化逻辑回归的代价函数

在这一部分，你将实现正则化逻辑回归的代价函数。

回想一下，对于正则化逻辑回归，代价函数的形式为
$$J(\mathbf{w},b) = \frac{1}{m}  \sum_{i=0}^{m-1} \left[ -y^{(i)} \log\left(f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) - \left( 1 - y^{(i)}\right) \log \left( 1 - f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) \right] + \frac{\lambda}{2m}  \sum_{j=0}^{n-1} w_j^2$$

将其与没有正则化的代价函数（你在上面实现的）进行比较，其形式为

$$ J(\mathbf{w}.b) = \frac{1}{m}\sum_{i=0}^{m-1} \left[ (-y^{(i)} \log\left(f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) - \left( 1 - y^{(i)}\right) \log \left( 1 - f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right)\right]$$

区别在于正则化项，即 $$\frac{\lambda}{2m}  \sum_{j=0}^{n-1} w_j^2$$
注意 $b$ 参数没有被正则化。

<a name='ex-05'></a>
### 练习 5

请完成下面的 `compute_cost_reg` 函数，为 $w$ 中的每个元素计算以下项
$$\frac{\lambda}{2m}  \sum_{j=0}^{n-1} w_j^2$$

然后启动代码将此添加到没有正则化的代价（你在上面的 `compute_cost` 中计算的）中，以计算带正则化的代价。

如果你遇到困难，可以查看下面单元格后面的提示来帮助你实现。

In [ ]:
# UNQ_C5
def compute_cost_reg(X, y, w, b, lambda_ = 1):
    """
    计算所有样本的代价
    参数:
      X : (array_like 形状 (m,n)) 数据，m 个样本，n 个特征
      y : (array_like 形状 (m,)) 目标值
      w : (array_like 形状 (n,)) 模型参数值
      b : (array_like 形状 (n,)) 模型的偏置参数值
      lambda_ : (标量, 浮点数)    控制正则化程度
    返回:
      total_cost: (标量)         代价
    """

    m, n = X.shape
    
    # 调用上面实现的 compute_cost 函数
    cost_without_reg = compute_cost(X, y, w, b) 
    
    # 你需要计算这个值
    reg_cost = 0.
    
    ### 开始编码 ###
    
        
    ### 结束编码 ### 
    
    # 添加正则化代价以获得总代价
    total_cost = cost_without_reg + (lambda_/(2 * m)) * reg_cost

    return total_cost

<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
    
    
* 以下是此函数的整体实现结构
    ```python 
       def compute_cost_reg(X, y, w, b, lambda_ = 1):
   
           m, n = X.shape
    
            # 调用上面实现的 compute_cost 函数
            cost_without_reg = compute_cost(X, y, w, b) 
    
            # 你需要计算这个值
            reg_cost = 0.
    
            ### 开始编码 ###
            for j in range(n):
                reg_cost_j = # 你的代码，计算 w[j] 的代价
                reg_cost = reg_cost + reg_cost_j

            ### 结束编码 ### 
    
            # 添加正则化代价以获得总代价
            total_cost = cost_without_reg + (lambda_/(2 * m)) * reg_cost

        return total_cost
    ```
  
    如果你仍然卡住，可以查看下面的提示来计算 `reg_cost_j`
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 reg_cost_j 的提示</b></font></summary>
           &emsp; &emsp; 你可以将 reg_cost_j 计算为 <code>reg_cost_j = w[j]**2 </code>
    </details>
        
    </details>

</details>

    

运行下面的单元格来检查你实现的 `compute_cost_reg` 函数。

In [ ]:
X_mapped = map_feature(X_train[:, 0], X_train[:, 1])
np.random.seed(1)
initial_w = np.random.rand(X_mapped.shape[1]) - 0.5
initial_b = 0.5
lambda_ = 0.5
cost = compute_cost_reg(X_mapped, y_train, initial_w, initial_b, lambda_)

print("Regularized cost :", cost)

# 单元测试
compute_cost_reg_test(compute_cost_reg)

**期望输出**：
<table>
  <tr>
    <td> <b>Regularized cost : <b></td>
    <td> 0.6618252552483948 </td> 
  </tr>
</table>

<a name="3.5"></a>
### 3.5 正则化逻辑回归的梯度

在本节中，你将实现正则化逻辑回归的梯度。


正则化代价函数的梯度有两个分量。第一个，$\frac{\partial J(\mathbf{w},b)}{\partial b}$ 是标量，另一个是与参数 $\mathbf{w}$ 形状相同的向量，其中第 $j^\mathrm{th}$ 个元素定义如下：

$$\frac{\partial J(\mathbf{w},b)}{\partial b} = \frac{1}{m}  \sum_{i=0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})  $$

$$\frac{\partial J(\mathbf{w},b)}{\partial w_j} = \left( \frac{1}{m}  \sum_{i=0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) x_j^{(i)} \right) + \frac{\lambda}{m} w_j  \quad\, \mbox{for $j=0...(n-1)$}$$

将其与没有正则化的代价函数的梯度（你在上面实现的）进行比较，其形式为
$$
\frac{\partial J(\mathbf{w},b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - \mathbf{y}^{(i)}) \tag{2}
$$
$$
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - \mathbf{y}^{(i)})x_{j}^{(i)} \tag{3}
$$


如你所见，$\frac{\partial J(\mathbf{w},b)}{\partial b}$ 是相同的，区别在于 $\frac{\partial J(\mathbf{w},b)}{\partial w}$ 中的以下项，即 $$\frac{\lambda}{m} w_j  \quad\, \mbox{for $j=0...(n-1)$}$$





<a name='ex-06'></a>
### 练习 6

请完成下面的 `compute_gradient_reg` 函数，修改下面的代码以计算以下项

$$\frac{\lambda}{m} w_j  \quad\, \mbox{for $j=0...(n-1)$}$$

启动代码将此项添加到上面 `compute_gradient` 返回的 $\frac{\partial J(\mathbf{w},b)}{\partial w}$ 中，以获得正则化代价函数的梯度。


如果你遇到困难，可以查看下面单元格后面的提示来帮助你实现。

In [ ]:
# UNQ_C6
def compute_gradient_reg(X, y, w, b, lambda_ = 1): 
    """
    计算带正则化的逻辑回归的梯度
 
    参数:
      X : (ndarray 形状 (m,n))   变量，如房屋大小
      y : (ndarray 形状 (m,))    实际值
      w : (ndarray 形状 (n,))    模型参数值
      b : (标量)                模型参数值
      lambda_ : (标量,浮点数)    正则化常数
    返回:
      dj_db: (标量)             代价关于参数 b 的梯度。
      dj_dw: (ndarray 形状 (n,)) 代价关于参数 w 的梯度。

    """
    m, n = X.shape
    
    dj_db, dj_dw = compute_gradient(X, y, w, b)

    ### 开始编码 ###     
    
        
    ### 结束编码 ###         
        
    return dj_db, dj_dw

<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
    
    
* 以下是此函数的整体实现结构
    ```python 
    def compute_gradient_reg(X, y, w, b, lambda_ = 1): 
        m, n = X.shape
    
        dj_db, dj_dw = compute_gradient(X, y, w, b)

        ### 开始编码 ###     
        # 遍历 w 的元素
        for j in range(n): 
            
            dj_dw_j_reg = # 你的代码，计算 dj_dw[j] 的正则化项
            
            # 将正则化项添加到 dj_dw 的相应元素
            dj_dw[j] = dj_dw[j] + dj_dw_j_reg
        
        ### 结束编码 ###         
        
        return dj_db, dj_dw
    ```
  
    如果你仍然卡住，可以查看下面的提示来计算 `dj_dw_j_reg`
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 dj_dw_j_reg 的提示</b></font></summary>
           &emsp; &emsp; 你可以将 dj_dw_j_reg 计算为 <code>dj_dw_j_reg = (lambda_ / m) * w[j] </code>
    </details>
        
    </details>

</details>

    


运行下面的单元格来检查你实现的 `compute_gradient_reg` 函数。

In [ ]:
X_mapped = map_feature(X_train[:, 0], X_train[:, 1])
np.random.seed(1) 
initial_w  = np.random.rand(X_mapped.shape[1]) - 0.5 
initial_b = 0.5
 
lambda_ = 0.5
dj_db, dj_dw = compute_gradient_reg(X_mapped, y_train, initial_w, initial_b, lambda_)

print(f"dj_db: {dj_db}", )
print(f"First few elements of regularized dj_dw:\n {dj_dw[:4].tolist()}", )

# 单元测试
compute_gradient_reg_test(compute_gradient_reg)

**期望输出**：
<table>
  <tr>
    <td> <b>dj_db:</b>0.07138288792343656</td> </tr>
  <tr>
      <td> <b> First few elements of regularized dj_dw:</b> </td> </tr>
   <tr>
   <td> [[-0.010386028450548701], [0.01140985288328012], [0.0536273463274574], [0.003140278267313462]] </td> 
  </tr>
</table>

<a name="3.6"></a>
### 3.6 使用梯度下降学习参数

与前面部分类似，你将使用上面实现的梯度下降函数来学习最优参数 $w$、$b$。
- 如果你正确完成了正则化逻辑回归的代价和梯度，你应该能够逐步执行下一个单元格来学习参数 $w$。
- 训练参数后，我们将使用它来绘制决策边界。

**注意**

下面的代码块需要相当长的时间才能运行，特别是非向量化版本。你可以减少 `iterations` 来测试你的实现并更快地迭代。如果你有时间，运行 100,000 次迭代以查看更好的结果。

In [ ]:
# 初始化拟合参数
np.random.seed(1)
initial_w = np.random.rand(X_mapped.shape[1])-0.5
initial_b = 1.

# 设置正则化参数 lambda_（你可以尝试改变它）
lambda_ = 0.01    

# 梯度下降设置
iterations = 10000
alpha = 0.01

w,b, J_history,_ = gradient_descent(X_mapped, y_train, initial_w, initial_b, 
                                    compute_cost_reg, compute_gradient_reg, 
                                    alpha, iterations, lambda_)

<details>
<summary>
    <b>期望输出：代价 < 0.5（点击查看详情）</b>
</summary>

```
# 使用以下设置
#np.random.seed(1)
#initial_w = np.random.rand(X_mapped.shape[1])-0.5
#initial_b = 1.
#lambda_ = 0.01;                                          
#iterations = 10000
#alpha = 0.01
Iteration    0: Cost     0.72   
Iteration 1000: Cost     0.59   
Iteration 2000: Cost     0.56   
Iteration 3000: Cost     0.53   
Iteration 4000: Cost     0.51   
Iteration 5000: Cost     0.50   
Iteration 6000: Cost     0.48   
Iteration 7000: Cost     0.47   
Iteration 8000: Cost     0.46   
Iteration 9000: Cost     0.45   
Iteration 9999: Cost     0.45       
    
```

<a name="3.7"></a>
### 3.7 绘制决策边界
为了帮助你可视化此分类器学习到的模型，我们将使用我们的 `plot_decision_boundary` 函数来绘制分隔正例和负例的（非线性）决策边界。

- 在函数中，我们通过计算分类器在均匀间隔网格上的预测来绘制非线性决策边界，然后绘制预测从 y = 0 变为 y = 1 的等高线图。

- 学习参数 $w$、$b$ 后，下一步是绘制类似图 4 的决策边界。

<img src="images/figure 4.png"  width="450" height="450">

In [ ]:
plot_decision_boundary(w, b, X_mapped, y_train)

<a name="3.8"></a>
### 3.8 评估正则化逻辑回归模型

你将使用上面实现的 `predict` 函数来计算正则化逻辑回归模型在训练集上的准确率

In [ ]:
# 计算训练集上的准确率
p = predict(X_mapped, w, b)

print('Train Accuracy: %f'%(np.mean(p == y_train) * 100))

**期望输出**：
<table>
  <tr>
    <td> <b>Train Accuracy:</b>~ 80%</td> </tr>
</table>

**恭喜你完成了本课程的最后一个实验！我们希望在课程 2 中见到你，届时你将使用更高级的学习算法，如神经网络和决策树。继续学习！**

<details>
  <summary><font size="2" color="darkgreen"><b>如果你想尝试任何非评分代码，请点击这里。</b></font></summary>
    <p><i><b>重要提示：请仅在通过作业后再执行此操作，以避免自动评分器出现问题。</b></i>
    <ol>
        <li> 在笔记本菜单上，点击"查看" > "单元格工具栏" > "编辑元数据"</li>
        <li> 点击你想要锁定/解锁的代码单元格旁边的"编辑元数据"按钮</li>
        <li> 将"editable"的属性值设置为：
            <ul>
                <li> 如果你想解锁它，设置为 "true"</li>
                <li> 如果你想锁定它，设置为 "false"</li>
            </ul>
        </li>
        <li> 在笔记本菜单上，点击"查看" > "单元格工具栏" > "无"</li>
    </ol>
    <p> 以下是上述步骤的简短演示：
        <br>
        <img src="https://drive.google.com/uc?export=view&id=14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" align="center" alt="unlock_cells.gif">
</details>